# 3D Point-Cloud Walkthrough

Phase 5: point-cloud **classification** and **part segmentation** (PointNet,
PointNet++, DGCNN) and **3D object detection** (PointPillars) — all from
scratch on pure-PyTorch point ops (FPS / ball-query / kNN), and all runnable
offline on CPU against synthetic primitive fixtures.

CUDA-only methods (SECOND, CenterPoint, BEVFormer, Mask3D) are registered as
gated wrappers that raise a clear error here and run on a Linux/CUDA box.

In [ ]:
import matplotlib.pyplot as plt
import torch
torch.manual_seed(0)

## 1. Synthetic primitives

`synthetic_pointcloud` samples points from cube / sphere / plane / cylinder
surfaces with random pose and jitter — deterministic per index.

In [ ]:
from image_analytics.data.datasets import build_dataset
from image_analytics.core.config import DataConfig

ds = build_dataset(DataConfig(dataset='synthetic_pointcloud',
                              kwargs={'size': 64, 'num_points': 1024}), split='train')
print('classes:', ds.CLASSES)

fig = plt.figure(figsize=(12, 3))
for col in range(4):
    pts, label = ds[col]
    ax = fig.add_subplot(1, 4, col + 1, projection='3d')
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=2)
    ax.set_title(ds.CLASSES[label]); ax.set_axis_off()
plt.tight_layout(); plt.show()

## 2. Train PointNet (classification)

PointNet aligns the input with a T-Net, extracts per-point features, and
max-pools to a global descriptor. The feature T-Net is regularised toward
orthogonality (added automatically by the point-cloud trainer).

In [ ]:
from image_analytics.core.config import load_config
from image_analytics.detection_3d.train import run

config = load_config('../configs/pointcloud/pointnet_primitives.yaml', overrides=[
    'training.epochs=8', 'training.device=cpu', 'training.warmup_epochs=0',
    'data.kwargs.size=256', 'output_dir=outputs',
])
metrics = run(config)
print('accuracy:', round(metrics['val/accuracy'], 3))

## 3. PointNet++ and DGCNN

`pointnet2` adds hierarchical Set Abstraction (FPS → ball query → mini-PointNet)
for local geometry; `dgcnn` builds dynamic kNN graphs and runs EdgeConv. Both
swap in by name and support `task: classification` or `task: segmentation`:

```bash
python scripts/train.py --config configs/pointcloud/dgcnn_primitives.yaml
python scripts/train.py --config configs/pointcloud/pointnet2_seg.yaml   # part seg
```

In [ ]:
from image_analytics.core.registry import MODELS
import image_analytics.detection_3d  # register

# Per-point segmentation logits are (B, num_parts, N).
seg = MODELS.build('pointnet2', num_classes=4, task='segmentation').eval()
with torch.no_grad():
    out = seg(torch.rand(1, 1024, 3))
print('seg logits:', tuple(out.shape))

## 4. 3D detection with PointPillars

Points are scattered into a BEV pseudo-image (pillar feature net), a 2D CNN
backbone runs over it, and an anchor-free head predicts class + box per cell.
The synthetic fixture is a ground plane with cuboid clusters and 3D boxes:

```bash
python scripts/train.py --config configs/pointcloud/pointpillars_synthetic.yaml
```

In [ ]:
det = build_dataset(DataConfig(dataset='synthetic_pointcloud_det',
                               kwargs={'size': 4, 'num_points': 2048}), split='train')
pts, target = det[0]
print('boxes_3d (x,y,z,dx,dy,dz,yaw):')
print(target['boxes_3d'])

# BEV scatter with GT box footprints
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(pts[:, 0], pts[:, 1], s=1, alpha=0.4)
for b in target['boxes_3d']:
    x, y, dx, dy = b[0], b[1], b[3], b[4]
    rect = plt.Rectangle((x - dx / 2, y - dy / 2), dx, dy, fill=False, color='r')
    ax.add_patch(rect)
ax.set_title('BEV + GT boxes'); ax.set_aspect('equal'); plt.show()

In [ ]:
# Forward a (random-init) PointPillars to see the prediction interface.
pp = MODELS.build('pointpillars', num_classes=1).eval()
with torch.no_grad():
    pred = pp(pts.unsqueeze(0))[0]
print('predicted boxes:', tuple(pred['boxes_3d'].shape),
      '| scores:', tuple(pred['scores'].shape))
print('(train with the config above for meaningful detections)')

## 5. CUDA-gated methods

SECOND / CenterPoint (spconv), BEVFormer (mmdet3d), and Mask3D
(MinkowskiEngine) are registered but require CUDA — they raise an actionable
error on CPU and build on a Linux/CUDA box via the `[3d-cuda]` extra:

```python
MODELS.build('centerpoint', num_classes=3)
# RuntimeError: CenterPoint requires CUDA ... install 'image-analytics[3d-cuda]'
```

## Summary

| Task | Models | Metric |
|---|---|---|
| Classification | `pointnet`, `pointnet2`, `dgcnn` | accuracy |
| Part segmentation | `pointnet2`, `pointnet`, `dgcnn` (`task: segmentation`) | mIoU |
| 3D detection | `pointpillars` | 3D mAP (`Detection3DEvaluator`) |
| CUDA-only | `second`, `centerpoint`, `bevformer`, `mask3d` | — (gated) |

This completes Phase 5. The pure-PyTorch point ops keep everything CPU-testable;
the CUDA wrappers slot in behind the same registry on a GPU box.